In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from PIL import Image, ImageDraw, ImageFont
import pandas as pd
import cyanomembranes as cm
from matplotlib.collections import LineCollection

In [ ]:
PIC_OUT = Path("../output/pub_fig")
PIC_OUT.mkdir(exist_ok=True)

prtoein_dispatch = {
    "3WU2-PSII-ThermosynVul": "PSII",
    "4H13-cytb6f": r"Cyt $\mathrm{b_6f}$",
    "1JB0-PSI-syn-cocc": "PSI",
    "avg_membrane": "Avg"
}

## Fig 1

In [ ]:
img1 = Image.open("../output/pictures_membranes/psi_crystals_membrane.png")
w, h = img1.size

sub_w = w // 3
sub_h = h // 3

left   = 2 * sub_w
top    = 2 * sub_h + 80
right  = w
bottom = h - 50

cropped = img1.crop((left, top, right, bottom))

img1 = Image.open("../output/pictures_membranes/psi_crystals_membrane.png")
w, h = img1.size

sub_w = w // 3
sub_h = h // 3

left   = 2 * sub_w
top    = 2 * sub_h + 80
right  = w
bottom = h - 50

cropped = img1.crop((left, top, right, bottom))

img2 = Image.open("../output/big_MD_fpt/picture/membrane_fpt_avg_membrane_1JB0-PSI-syn-cocc_mv1.2.png")
w, h = img2.size

sub_w = w // 3

cropped2 = img2.crop((2*sub_w, 0, w, h))

img3 = Image.open("../output/big_MD_rate/picture/rate_constant.png")
img4 = Image.open("../output/crystal_analysis/individual_runs/1JB0-PSI-syn-cocc_hexagonal_1.2.csv_run0.png")

images = [cropped, img4, cropped2, img3]
target_w = images[0].width
images = [img.resize((target_w, int(img.height * target_w / img.width)), Image.LANCZOS) for img in images]

img1 = images[1]
images[1] = images[1].resize((target_w, int(img1.height * target_w / img1.width)), Image.LANCZOS)

W = target_w
H_top = images[0].height
H_bot = max(images[2].height, images[3].height)
margin_top = 50

canvas = Image.new("RGB", (2 * W, H_top + H_bot + margin_top), "white")

try:
    font = ImageFont.truetype("DejaVuSans.ttf", 48)
except:
    font = ImageFont.load_default()

draw = ImageDraw.Draw(canvas)

def draw_label(draw, label, x, y, font):
    for dx, dy in [(-2,0),(2,0),(0,-2),(0,2),(-2,-2),(2,-2),(-2,2),(2,2)]:
        draw.text((x + dx, y + dy), label, fill="white", font=font)
    draw.text((x, y), label, fill="black", font=font)

# Upper row
for img, col, label in zip(images[:2], [0, 1], ["a)", "b)"]):
    x = col * W
    y = margin_top
    canvas.paste(img, (x, y))
    draw_label(draw, label, x + 15, y - 30, font)

# Lower row
label_y_bot = margin_top + H_top + 15
offsets = [0, -30]
for img, col, label, offset in zip(images[2:], [0, 1], ["c)", "d)"], offsets):
    x = col * W
    y = margin_top + H_top + (H_bot - img.height) + offset
    canvas.paste(img, (x, y))
    draw_label(draw, label, x + 15, label_y_bot, font)

canvas

## Fig 2

In [ ]:
files = [
    "../output/no_MD_diff_random_start/picture/membrane_diffusion_avg_membrane.png",
    "../output/no_MD_diff_random_start/picture/membrane_diffusion_last_D_dist.png"
]

target_height = 600
imgs = []

for f in files:
    img = Image.open(f)
    w, h = img.size

    new_w = int(w * target_height / h)
    imgs.append(img.resize((new_w, target_height), Image.LANCZOS))

total_width = sum(img.size[0] for img in imgs)
combined = Image.new("RGB", (total_width, target_height), "white")

x_offset = 0
positions = []

for img in imgs:
    combined.paste(img, (x_offset, 0))
    positions.append(x_offset)
    x_offset += img.size[0]

draw = ImageDraw.Draw(combined)

labels = ["a)", "b)", "c)"]

try:
    font = ImageFont.truetype("DejaVuSans.ttf", 30)  # large bold font
except:
    font = ImageFont.load_default()

for i, x in enumerate(positions):
    draw.text((x + 15, 15), labels[i], fill="black", font=font)
    if i==1:
        draw.text((x + 715, 15), labels[i + 1], fill="black", font=font)

combined.save(PIC_OUT / "Fig2.png")

In [ ]:
df = pd.read_csv("../output/no_MD_diff_random_start/picture/reg_param_df.csv", index_col=0)
df.pkey = df.pkey.apply(lambda x: prtoein_dispatch[x])

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')

tbl = ax.table(
    cellText=df.round(3).values,
    colLabels=["Protein", "Type", "Slope", "Intercept", "R²"],
    cellLoc='center',
    loc='center'
)

tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.3, 1.5)

# styling header
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_fontsize(12)
        cell.set_text_props(weight='bold')
        cell.set_facecolor("#f2f2f2")

plt.savefig(PIC_OUT / "Fig2_table.png", dpi=300, bbox_inches='tight')

## Fig 3

In [ ]:
files = [
    "../output/no_MD_fpt/picture/3d_membrane_fpt_avg_membrane.png",
    "../output/pictures_membranes/Medial_Axis_hist_avg_membrane_improved.png",
    "../output/pictures_membranes/all_rdf_average_membrane.png"
]

arget_height = 600
imgs = []

for f in files:
    img = Image.open(f)
    w, h = img.size

    new_w = int(w * target_height / h)
    imgs.append(img.resize((new_w, target_height), Image.LANCZOS))

total_width = sum(img.size[0] for img in imgs)
combined = Image.new("RGB", (total_width, target_height), "white")

x_offset = 0
positions = []

for img in imgs:
    combined.paste(img, (x_offset, 0))
    positions.append(x_offset)
    x_offset += img.size[0]

draw = ImageDraw.Draw(combined)

labels = ["a)", "b)", "c)"]

try:
    font = ImageFont.truetype("DejaVuSans.ttf", 30)  # large bold font
except:
    font = ImageFont.load_default()

for i, x in enumerate(positions):
    draw.text((x + 15, 15), labels[i], fill="black", font=font)

combined.save(PIC_OUT / "Fig3.png")

# Table Methods 

In [ ]:
cols = ["Complex", "PDB ID", "Organism"]
rows = [
    ("PSII",                     "3WU2", r"$\mathit{Thermostichus\ vulcanus}$"),
    ("PSI",                      "1JB0", r"$\mathit{Synechococcus\ elongatus}$"),
    (r"$\mathrm{Cyt\ b_{6}f}$", "4H13", r"$\mathit{Mastigocladus\ laminosus}$"),
]
df = pd.DataFrame(rows, columns=cols)

fig, ax = plt.subplots(figsize=(10, 3))
ax.axis('off')
tbl = ax.table(
    cellText=df.values,
    colLabels=cols,
    cellLoc='center',
    loc='center'
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(11)
tbl.scale(1.3, 1.5)

col_widths = [0.2, 0.15, 0.3]
for (row, col), cell in tbl.get_celld().items():
    cell.set_width(col_widths[col])
    if row == 0:
        cell.set_fontsize(12)
        cell.set_text_props(weight='bold')
        cell.set_facecolor("#f2f2f2")

# Crop to table
fig.canvas.draw()
bbox = tbl.get_window_extent(fig.canvas.get_renderer())
bbox = bbox.transformed(fig.dpi_scale_trans.inverted())
bbox = bbox.expanded(1.02, 1.1)  # expand slightly horizontally and vertically
fig.savefig(PIC_OUT / "table_method.png", dpi=150, bbox_inches=bbox)

# Supplement figures

In [ ]:
files = [
    "../output/no_MD_diff_random_start/picture/membrane_diffusion_1JB0-PSI-syn-cocc.png",
    "../output/no_MD_diff_random_start/picture/membrane_diffusion_3WU2-PSII-ThermosynVul.png",
    "../output/no_MD_diff_random_start/picture/membrane_diffusion_4H13-cytb6f.png"
]


target_height = 600
top_margin = 80  # space above plots

imgs = []

# --- resize images ---
for f in files:
    img = Image.open(f)
    w, h = img.size

    new_w = int(w * target_height / h)
    imgs.append(img.resize((new_w, target_height), Image.LANCZOS))

# --- canvas size ---
total_width = sum(img.size[0] for img in imgs)

combined = Image.new(
    "RGB",
    (total_width, target_height + top_margin),
    "white"
)

# --- paste images shifted down ---
x_offset = 0
positions = []

for img in imgs:
    combined.paste(img, (x_offset, top_margin))
    positions.append(x_offset)
    x_offset += img.size[0]

# --- draw labels ---
draw = ImageDraw.Draw(combined)

labels = ["a) PSI", "b) PSII", "c) Cyt b6f"]

try:
    font = ImageFont.truetype("DejaVuSans.ttf", 30)
except:
    font = ImageFont.load_default()

for i, x in enumerate(positions):
    draw.text(
        (x + 10, top_margin // 3),  # centered in margin
        labels[i],
        fill="black",
        font=font
    )

# --- save ---
combined.save(PIC_OUT / "Sup-Fig_No_MD_Diff.png")

## Protein Shadows

In [ ]:

projected_coords, edge_points, concave_hull = cm.pdb_utils.make_polygon("../output/protein_shadows/1JB0-PSI-syn-cocc_wo.pdb", out_file=None)

fig,axes = plt.subplots(1,3, figsize=(15,5))
axes[0].scatter(projected_coords[:,0],projected_coords[:,1], s=1, alpha=0.5)

for spine in axes[0].spines.values():
    spine.set_visible(False)

axes[0].set_xticks([])
axes[0].set_yticks([])
axes[0].set_aspect("equal")
axes[0].set_title("Projected coord", size=20)


lc = LineCollection(edge_points, linewidth=0.2)

for spine in axes[1].spines.values():
    spine.set_visible(False)

axes[1].add_collection(lc)
axes[1].set_aspect("equal")
axes[1].autoscale()
axes[1].set_xticks([])
axes[1].set_yticks([])
axes[1].set_title("Triangulation", size=20)

axes[2].fill(*concave_hull.exterior.xy, lw=1, alpha=0.5, edgecolor="blue")

for spine in axes[2].spines.values():
    spine.set_visible(False)

axes[2].set_aspect("equal")
axes[2].set_xticks([])
axes[2].set_yticks([])
axes[2].set_title("Concave hull", size=20)

plt.tight_layout()
fig.savefig(PIC_OUT / "Sup-ProteinShadow.png", dpi=300, bbox_inches="tight")
plt.show()

## Periodic boundaries

In [ ]:
p = cm.geo_utils.readwkt("../output/avg_membrane/polygons_avg_membrane_110-9.wkt")
fig, ax = plt.subplots()
for i in p:
    ax.plot(*i.exterior.xy, lw=0.1, c="blue")
ax.set_aspect("equal")
ax.vlines(0,-5000,10000)
ax.vlines(5000,-5000,10000)
ax.hlines(0,-5000,10000)
ax.hlines(5000,-5000,10000)
ax.set_ylabel(r"$\mathrm{\AA}$")
ax.set_xlabel(r"$\mathrm{\AA}$")
fig.savefig(PIC_OUT / "Sup-Periodic_boundaries.png", dpi=300)
plt.show()

## Crystal analysis

In [ ]:
files = [
    "../output/Crystal_analysis/crystal_survival_1JB0-PSI-syn-cocc_hexagonal_ps-5.png",
    "../output/Crystal_analysis/crystal_survival_1JB0-PSI-syn-cocc_square_ps-5.png",
    "../output/Crystal_analysis/crystal_survival_3WU2-PSII-ThermosynVul_hexagonal_ps-5.png",
    "../output/Crystal_analysis/crystal_survival_3WU2-PSII-ThermosynVul_square_ps-5.png",
]

label_font_size = 25       
label_offset = (10, 5)     
label_color = (0, 0, 0)    
padding = 20              

imgs = [Image.open(p).convert("RGB") for p in files]

w, h = imgs[0].size 

font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", label_font_size)

canvas_w = 2 * w + 3 * padding
canvas_h = 2 * h + 3 * padding
canvas = Image.new("RGB", (canvas_w, canvas_h), color=(255, 255, 255))

labels = ["a)", "b)", "c)", "d)"]
positions = [
    (padding,         padding),          # A: top-left
    (2 * padding + w, padding),          # B: top-right
    (padding,         2 * padding + h),  # C: bottom-left
    (2 * padding + w, 2 * padding + h),  # D: bottom-right
]

draw = ImageDraw.Draw(canvas)

for img, label, (x, y) in zip(imgs, labels, positions):
    canvas.paste(img, (x, y))
    draw.text(
        (x + label_offset[0], y + label_offset[1]),
        label,
        font=font,
        fill=label_color,
    )

# --- Save ---
canvas.save(PIC_OUT / "Sup-Crystalanalysis.png", dpi=(300, 300))
